# kaggle-vllm 0.2.0 post-publication acceptance

Output-free final release-gate notebook. Run only in a fresh Kaggle **GPU T4 x2** session with Internet enabled, after `kaggle-vllm==0.2.0` exists on PyPI. A valid notebook file or local CPU run is not acceptance evidence.

In [1]:
from pathlib import Path
from importlib import metadata
import gc
import json
import os
import platform
import signal
import subprocess
import sys
import time
import urllib.error
import urllib.request

EXPECTED_SDK_VERSION = "0.2.0"
MODEL_REPO = "facebook/opt-125m"
MODEL_REVISION = "27dcfa74d334bc871f3234de431e71c6eeba5dd6"
WORK = Path("/kaggle/working/kaggle-vllm-020-published")
RUNTIME = WORK / "runtime"
STAGED = RUNTIME / "vllm-staged"
OVERLAY = RUNTIME / "vllm-runtime-overlay"
MANIFEST = RUNTIME / "kaggle-vllm-runtime.json"
CACHE = Path("/kaggle/working/kaggle-vllm-cache")
EVIDENCE = WORK / "kaggle-vllm-020-published-acceptance-evidence.json"
WORK.mkdir(parents=True, exist_ok=True)

print("Python:", sys.version)
print("Platform:", platform.platform())
assert sys.version_info[:3] == (3, 12, 13), sys.version
subprocess.run(["nvidia-smi", "-L"], check=True)
subprocess.run(["nvidia-smi", "topo", "-m"], check=True)
nvcc = subprocess.check_output(["nvcc", "--version"], text=True)
assert "release 12.8" in nvcc, nvcc

import torch
torch_before = {
    "version": torch.__version__,
    "cuda": torch.version.cuda,
    "path": str(Path(torch.__file__).resolve()),
}
assert torch_before["version"] == "2.10.0+cu128", torch_before
assert torch_before["cuda"] == "12.8", torch_before
assert torch.cuda.device_count() == 2
assert all(torch.cuda.get_device_name(i) == "Tesla T4" for i in range(2))
assert all(torch.cuda.get_device_capability(i) == (7, 5) for i in range(2))
print("Torch before bootstrap:", torch_before)


Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.12.90+-x86_64-with-glibc2.35
GPU 0: Tesla T4 (UUID: GPU-a3348fbf-5c81-e767-5eb9-daf5aa863825)
GPU 1: Tesla T4 (UUID: GPU-285fd72b-d8d6-4acd-b32a-ff84e7aac0df)
	GPU0	GPU1	CPU Affinity	NUMA Affinity	GPU NUMA ID
GPU0	 X 	PHB	0-3	0		N/A
GPU1	PHB	 X 	0-3	0		N/A

Legend:

  X    = Self
  SYS  = Connection traversing PCIe as well as the SMP interconnect between NUMA nodes (e.g., QPI/UPI)
  NODE = Connection traversing PCIe as well as the interconnect between PCIe Host Bridges within a NUMA node
  PHB  = Connection traversing PCIe as well as a PCIe Host Bridge (typically the CPU)
  PXB  = Connection traversing multiple PCIe bridges (without traversing the PCIe Host Bridge)
  PIX  = Connection traversing at most a single PCIe bridge
  NV#  = Connection traversing a bonded set of # NVLinks
Torch before bootstrap: {'version': '2.10.0+cu128', 'cuda': '12.8', 'path': '/usr/local/lib/python3.12/dist-packages/torch/__init__.

## Install the exact public SDK without native-runtime dependencies

In [2]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-cache-dir", f"kaggle-vllm[hub]=={EXPECTED_SDK_VERSION}"],
    check=True,
)
import kaggle_vllm
from packaging.requirements import Requirement
assert kaggle_vllm.__version__ == EXPECTED_SDK_VERSION
requirements = metadata.requires("kaggle-vllm") or []
required_names = {Requirement(item).name.casefold() for item in requirements}
forbidden = {"torch", "torchvision", "torchaudio", "vllm", "nvidia-cuda-runtime-cu12"}
assert not (required_names & forbidden), required_names & forbidden
print("Installed SDK:", metadata.version("kaggle-vllm"))
print("SDK Requires-Dist names:", sorted(required_names))


Installed SDK: 0.2.0
SDK Requires-Dist names: ['build', 'huggingface_hub', 'packaging', 'pytest', 'ruff', 'twine']


## Strict immutable bootstrap, activation, native imports, and doctor

In [3]:
BOOTSTRAP = [
    "kaggle-vllm", "bootstrap", "--strict",
    "--staged", str(STAGED), "--overlay", str(OVERLAY),
    "--cache", str(CACHE), "--manifest", str(MANIFEST),
]
subprocess.run(["kaggle-vllm", "fingerprint"], check=True)
subprocess.run(BOOTSTRAP + ["--dry-run", "--json"], check=True)
subprocess.run(BOOTSTRAP, check=True)
manifest = json.loads(MANIFEST.read_text(encoding="utf-8"))
assert manifest["wheel"]["filename"] == "vllm-0.18.2.dev0+ga26e8dc7f.d20260822.cu128-cp312-cp312-linux_x86_64.whl"
assert manifest["wheel"]["sha256"] == "5a9bd710b8a19fdd23abb3442baad892da977466f996334decd533a225f5fd0c"
assert manifest["wheel"]["hf_revision"] == "f6b4f10de54924ed6fe9e28cceab84eca7276ab6"

from kaggle_vllm import activate_runtime
assert activate_runtime(MANIFEST)
import vllm
import vllm._C
import vllm._moe_C
import vllm.cumem_allocator
for module in (vllm, vllm._C, vllm._moe_C, vllm.cumem_allocator):
    assert Path(module.__file__).resolve().is_relative_to(STAGED.resolve()), module.__file__
doctor = subprocess.run(["kaggle-vllm", "doctor", "--strict", "--json"], check=True, capture_output=True, text=True)
doctor_payload = json.loads(doctor.stdout)
assert doctor_payload["compatible"] is True
torch_after = {"version": torch.__version__, "cuda": torch.version.cuda, "path": str(Path(torch.__file__).resolve())}
assert torch_after == torch_before, (torch_before, torch_after)
print("Native imports, strict doctor, and Torch preservation: PASS")


{
  "is_kaggle": true,
  "python": "3.12.13",
  "platform": "Linux-6.12.90+-x86_64-with-glibc2.35",
  "torch": "2.10.0+cu128",
  "torch_path": "/usr/local/lib/python3.12/dist-packages/torch/__init__.py",
  "torch_cuda": "12.8",
  "cuda_available": true,
  "gpus": [
    {
      "index": 0,
      "name": "Tesla T4",
      "capability": [
        7,
        5
      ],
      "total_memory": 15636037632
    },
    {
      "index": 1,
      "name": "Tesla T4",
      "capability": [
        7,
        5
      ],
      "total_memory": 15636037632
    }
  ],
  "nccl": "2.27.5",
  "nvcc": "/usr/local/cuda/bin/nvcc",
  "nvcc_version": "nvcc: NVIDIA (R) Cuda compiler driver\nCopyright (c) 2005-2025 NVIDIA Corporation\nBuilt on Fri_Feb_21_20:23:50_PST_2025\nCuda compilation tools, release 12.8, V12.8.93\nBuild cuda_12.8.r12.8/compiler.35583870_0",
  "cuda_home": "/usr/local/cuda",
  "cuda_driver": "/usr/local/nvidia/lib64/libcuda.so",
  "cmake_library_path": null,
  "driver_version": "580.159.04",


Processing ./kaggle-vllm-cache/vllm-0.18.2.dev0+ga26e8dc7f.d20260822.cu128-cp312-cp312-linux_x86_64.whl
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 131.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.2/461.2 kB 384.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.6/192.6 kB 386.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 241.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.1/78.1 kB 343.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 808.1/808.1 kB 343.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 368.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.1/83.1 kB 286.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━

## Raw two-rank NCCL smoke

In [4]:
nccl_script = WORK / "nccl_smoke.py"
nccl_script.write_text(r'''import socket
import torch
import torch.distributed as dist
import torch.multiprocessing as mp

def worker(rank, world_size, port):
    torch.cuda.set_device(rank)
    dist.init_process_group("nccl", init_method=f"tcp://127.0.0.1:{port}", rank=rank, world_size=world_size)
    value = torch.tensor([float(rank + 1)], device=f"cuda:{rank}")
    dist.all_reduce(value)
    assert value.item() == 3.0, value
    print(f"rank={rank} all_reduce={value.item()}")
    dist.destroy_process_group()

if __name__ == "__main__":
    with socket.socket() as sock:
        sock.bind(("127.0.0.1", 0))
        port = sock.getsockname()[1]
    mp.spawn(worker, args=(2, port), nprocs=2, join=True)
''', encoding="utf-8")
subprocess.run([sys.executable, str(nccl_script)], check=True, env=os.environ.copy())
print("Raw NCCL smoke: PASS")


rank=1 all_reduce=3.0
rank=0 all_reduce=3.0
Raw NCCL smoke: PASS


## Immutable OPT-125M snapshot, TP=1, and TP=2

In [5]:
from huggingface_hub import snapshot_download
model_path = Path(snapshot_download(repo_id=MODEL_REPO, revision=MODEL_REVISION, cache_dir="/kaggle/working/huggingface"))
assert model_path.name == MODEL_REVISION
smoke = r'''from kaggle_vllm import KaggleLLM
from vllm import SamplingParams
import sys
model, tp = sys.argv[1], int(sys.argv[2])
llm = KaggleLLM(model=model, tensor_parallel_size=tp, dtype="float16", max_model_len=512, gpu_memory_utilization=0.40, enforce_eager=True, disable_custom_all_reduce=True)
out = llm.generate([f"kaggle-vllm public 0.2.0 TP={tp}:"], SamplingParams(temperature=0.0, max_tokens=32))
assert out and out[0].outputs and out[0].outputs[0].text
print(out[0].outputs[0].text)
'''
for tp in (1, 2):
    env = os.environ.copy()
    if tp == 1:
        env["CUDA_VISIBLE_DEVICES"] = "0"
    else:
        env.pop("CUDA_VISIBLE_DEVICES", None)
    subprocess.run([sys.executable, "-c", smoke, str(model_path), str(tp)], check=True, env=env)
    time.sleep(2)
print("OPT-125M TP=1 and TP=2: PASS")


Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/651 [00:00<?, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

flax_model.msgpack:   0%|          | 0.00/250M [00:00<?, ?B/s]

...

(EngineCore pid=319) INFO 08-31 11:26:49 [core.py:1201] Shutdown initiated (timeout=0)
(EngineCore pid=319) INFO 08-31 11:26:49 [core.py:1224] Shutdown complete
1:1:1:1:1:1:1:1:1:1:1:1:1:1:1:1:
ERROR 08-31 11:26:49 [core_client.py:704] Engine core proc EngineCore died unexpectedly, shutting down client.
INFO 08-31 11:27:09 [utils.py:233] non-default args: {'dtype': 'float16', 'max_model_len': 512, 'tensor_parallel_size': 2, 'gpu_memory_utilization': 0.4, 'disable_log_stats': True, 'enforce_eager': True, 'disable_custom_all_reduce': True, 'model': '/kaggle/working/huggingface/models--facebook--opt-125m/snapshots/27dcfa74d334bc871f3234de431e71c6eeba5dd6'}
INFO 08-31 11:27:09 [model.py:533] Resolved architecture: OPTForCausalLM
INFO 08-31 11:27:09 [model.py:1582] Using max model len 512
INFO 08-31 11:27:09 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 08-31 11:27:09 [vllm.py:775] Asynchronous scheduling is enabled.
WARNING 08-31 11:27:09 [vllm.py:809

Loading pt checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading pt checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  3.45it/s]
Loading pt checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  3.45it/s]
(Worker_TP0 pid=498) 


(Worker_TP0 pid=498) INFO 08-31 11:27:45 [default_loader.py:384] Loading weights took 0.30 seconds
(Worker_TP0 pid=498) INFO 08-31 11:27:46 [gpu_model_runner.py:4566] Model loading took 0.12 GiB memory and 0.346054 seconds
(Worker_TP0 pid=498) INFO 08-31 11:28:02 [gpu_worker.py:456] Available KV cache memory: 5.48 GiB
(EngineCore pid=474) INFO 08-31 11:28:02 [kv_cache_utils.py:1316] GPU KV cache size: 319,072 tokens
(EngineCore pid=474) INFO 08-31 11:28:02 [kv_cache_utils.py:1321] Maximum concurrency for 512 tokens per request: 623.19x
(EngineCore pid=474) INFO 08-31 11:28:03 [core.py:281] init engine (profile, create kv cache, warmup model) took 16.98 seconds
(EngineCore pid=474) INFO 08-31 11:28:05 [vllm.py:775] Asynchronous scheduling is enabled.
(EngineCore pid=474) WARNING 08-31 11:28:05 [vllm.py:809] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
(EngineCore pid=474) WARNING 08-31 11:28:05 [vllm.py:82

Processed prompts: 100%|██████████| 1/1 [00:04<00:00,  4.05s/it, est. speed input: 4.45 toks/s, output: 7.91 toks/s]


(EngineCore pid=474) INFO 08-31 11:28:09 [core.py:1201] Shutdown initiated (timeout=0)
(EngineCore pid=474) INFO 08-31 11:28:09 [core.py:1224] Shutdown complete
(Worker_TP0 pid=498) INFO 08-31 11:28:09 [multiproc_executor.py:759] Parent process exited, terminating worker queues
(Worker_TP0 pid=498) INFO 08-31 11:28:09 [multiproc_executor.py:854] WorkerProc shutting down.
(Worker_TP1 pid=499) INFO 08-31 11:28:09 [multiproc_executor.py:759] Parent process exited, terminating worker queues
(Worker_TP1 pid=499) INFO 08-31 11:28:09 [multiproc_executor.py:854] WorkerProc shutting down.
1

kaggle-vllm public 0.2.0 TP=2:1

kaggle-vllm public 0
ERROR 08-31 11:28:13 [core_client.py:704] Engine core proc EngineCore died unexpectedly, shutting down client.
OPT-125M TP=1 and TP=2: PASS


## Local OpenAI-compatible server and clean shutdown

In [6]:
server_log_path = WORK / "openai-server.log"
server_log = server_log_path.open("w", encoding="utf-8")
server = subprocess.Popen(
    ["kaggle-vllm", "serve", str(model_path), "--served-model-name", "opt-125m-kaggle-vllm-020", "--tensor-parallel-size", "2", "--max-model-len", "512", "--gpu-memory-utilization", "0.40", "--host", "127.0.0.1", "--port", "8000"],
    stdout=server_log, stderr=subprocess.STDOUT, text=True, env=os.environ.copy(), start_new_session=True,
)
models_payload = completion_payload = None
try:
    for _ in range(180):
        if server.poll() is not None:
            raise RuntimeError(f"server exited early with {server.returncode}")
        try:
            with urllib.request.urlopen("http://127.0.0.1:8000/v1/models", timeout=2) as response:
                assert response.status == 200
                models_payload = json.load(response)
                break
        except (urllib.error.URLError, TimeoutError):
            time.sleep(2)
    assert models_payload is not None
    body = json.dumps({"model": "opt-125m-kaggle-vllm-020", "prompt": "NCCL enables", "max_tokens": 16, "temperature": 0.0}).encode()
    request = urllib.request.Request("http://127.0.0.1:8000/v1/completions", data=body, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(request, timeout=120) as response:
        assert response.status == 200
        completion_payload = json.load(response)
    assert completion_payload.get("choices")
finally:
    if server.poll() is None:
        os.killpg(server.pid, signal.SIGTERM)
        try:
            server.wait(timeout=30)
        except subprocess.TimeoutExpired:
            os.killpg(server.pid, signal.SIGKILL)
            server.wait(timeout=10)
    server_log.close()
assert server.poll() is not None
gc.collect()
torch.cuda.empty_cache()
print("OpenAI models/completions HTTP 200 and clean process-group shutdown: PASS")


OpenAI models/completions HTTP 200 and clean process-group shutdown: PASS


## Machine-readable final result

In [7]:
evidence = {
    "status": "PASS",
    "sdk_version": kaggle_vllm.__version__,
    "python": platform.python_version(),
    "torch_before": torch_before,
    "torch_after": torch_after,
    "native_wheel": manifest["wheel"],
    "doctor_compatible": doctor_payload["compatible"],
    "model_repository": MODEL_REPO,
    "model_revision": MODEL_REVISION,
    "checks": {
        "environment_identity": True, "strict_bootstrap": True, "sha256_verified": True,
        "torch_preserved": True, "native_imports": True, "strict_doctor": True,
        "raw_nccl": True, "opt_tp1": True, "opt_tp2": True,
        "openai_models_http_200": True, "openai_completions_http_200": True,
        "clean_server_shutdown": True,
    },
}
EVIDENCE.write_text(json.dumps(evidence, indent=2) + "\n", encoding="utf-8")
print(json.dumps(evidence, indent=2))
print("FINAL PUBLISHED 0.2.0 ACCEPTANCE: PASS")


{
  "status": "PASS",
  "sdk_version": "0.2.0",
  "python": "3.12.13",
  "torch_before": {
    "version": "2.10.0+cu128",
    "cuda": "12.8",
    "path": "/usr/local/lib/python3.12/dist-packages/torch/__init__.py"
  },
  "torch_after": {
    "version": "2.10.0+cu128",
    "cuda": "12.8",
    "path": "/usr/local/lib/python3.12/dist-packages/torch/__init__.py"
  },
  "native_wheel": {
    "filename": "vllm-0.18.2.dev0+ga26e8dc7f.d20260822.cu128-cp312-cp312-linux_x86_64.whl",
    "sha256": "5a9bd710b8a19fdd23abb3442baad892da977466f996334decd533a225f5fd0c",
    "hf_repo_id": "waqasm86/kaggle-vllm-binaries",
    "hf_revision": "f6b4f10de54924ed6fe9e28cceab84eca7276ab6",
    "resolved_path": "/kaggle/working/kaggle-vllm-cache/vllm-0.18.2.dev0+ga26e8dc7f.d20260822.cu128-cp312-cp312-linux_x86_64.whl"
  },
  "doctor_compatible": true,
  "model_repository": "facebook/opt-125m",
  "model_revision": "27dcfa74d334bc871f3234de431e71c6eeba5dd6",
  "checks": {
    "environment_identity": true,
    "st